In [ ]:
from dotenv import load_dotenv
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.window import Window
import polars as pl
import pyspark
import pyspark.sql.functions as F
import urllib.request

load_dotenv();

In [ ]:
JDBC_OPTIONS = {
    "url": (
        f"jdbc:postgresql://{os.environ['DB_HOST']}:"
        f"{os.environ.get('DB_PORT', '5432')}/{os.environ['DB_NAME']}"
    ),
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

In [ ]:
def get_spark(app_name: str = "Postgres") -> SparkSession:
    return (
        SparkSession.builder
        .config("spark.sql.repl.eagerEval.enabled", "True")

        # Load jdbc driver jar from local path location
        # .config("spark.jars", str(Path(os.environ["PG_JDBC_DRIVER_PATH"])))

        # or download the jdbc driver directly from Maven repository
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
        .appName(app_name)
        .getOrCreate()
    )

In [ ]:
def read_query(spark: SparkSession, sql: str, **extra_options) -> DataFrame:
    """Run a SQL query against Postgres and return a Spark DataFrame."""
    return (
        spark.read.format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("query", sql)
        .options(**extra_options)
        .load()
    )

In [ ]:
def read_table(spark: SparkSession, table: str, **extra_options) -> DataFrame:
    """Read a whole table (Spark pushes down filters where it can)."""
    return (
        spark.read.format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("dbtable", table)
        .options(**extra_options)
        .load()
    )

In [ ]:
def write_dataframe(df: DataFrame, table: str, mode: str = "overwrite") -> None:
    """Write a PySpark DataFrame to a Postgres table."""
    (
        df
        .write
        .format("jdbc")
        .options(**JDBC_OPTIONS)
        .option("dbtable", table)
        .mode(mode)
        .save()
    )

In [ ]:
spark = get_spark()

In [ ]:
query = """
SELECT year, make, model
FROM public.vehicles limit 5
"""

In [ ]:
sdf = read_query(spark, query)

In [ ]:
sdf.show()

In [ ]:
write_dataframe(spark, sdf, "public.new_table")

In [ ]:
def to_polars(self) -> pl.DataFrame:
    """Collect a Spark DataFrame to the driver as a Polars DataFrame."""
    if hasattr(self, "toArrow"):          # Spark 4.0+
        return pl.from_arrow(self.toArrow())
    return pl.from_pandas(self.toPandas())  # Spark 3.x fallback

DataFrame.toPolars = to_polars

In [ ]:
# Download csv into memory (nothing is written to disk)
text = urllib.request.urlopen("https://www.fueleconomy.gov/feg/epadata/vehicles.csv").read().decode("utf-8")

# Turn the lines into an RDD of strings, then let Spark parse them as CSV
lines = text.splitlines()
vehicles = spark.read.csv(spark.sparkContext.parallelize(lines), header=True, inferSchema=True, sep=",")

In [ ]:
(
    vehicles
    .select(
        F.col("year")
        , F.col("make")
        , F.col("model")
        , F.col("basemodel")
        , F.col("fuelType")
        , F.col("fuelType1")
        , F.col("fuelType2")
        , F.col("atvtype")
        , F.col("charge240").alias("hrs_to_fully_charge_240v")
        , F.col("city08").alias("city_mpg")
        , F.col("comb08").alias("combined_mpg")
    )
).show(5)

In [ ]:
spark.stop()

In [ ]:
query = """
SELECT year, make, model
FROM public.vehicles limit 5
"""

In [ ]:
jdbcDF = (
    spark.read
    .format("jdbc")
    .option("driver", driver)
    .option("url", url)
    .option("user", PG_USER)
    .option("password", PG_PWD)
    .option("query", query)
    .load()
)

In [ ]:
jdbcDF.show()

In [ ]:
type(spark.sql)

In [ ]:
def spark_frame(query: str) -> pyspark.sql.DataFrame:
    

In [ ]:
import pyspark

In [ ]:
pyspark.sql.DataFrame